# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aishwarya00608/FlyRank_Assignment1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

## 1. My Lane (or Freestyle) and Why

**Chosen Lane:** Lane 1 — Content Decay & Refresh Prioritization (Predicting organic traffic decay to prioritize editorial updates).

**Why this lane:**  
Content updates require expensive editorial human hours. In SEO and organic search management, teams often update pages blindly (e.g., "anything older than 6 months") or reactively (after impressions have already crashed). Predicting which URLs are actively entering terminal decay versus normal seasonal plateaus allows content teams to intervene proactively while the page still retains historical link equity and search authority.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## 2. The Question: Decision, Action, Cost of a Wrong Call

* **Research Question:**  
  *Can we predict whether an established, high-traffic URL will experience a sustained drop of ≥25% in organic impressions over the next 28 days based on recent click-through trends, ranking position shifts, and content age?*

* **Unit of Analysis:**  
  A **URL-window pair** (specifically, a single URL evaluated over a rolling 28-day feature observation window to predict the subsequent 28-day performance window).

* **Model Output:**  
  A calibrated risk score ($P(\text{decay}) \in [0, 1]$), mapped into three actionable priority tiers: **High Risk (Immediate Refresh)**, **Medium Risk (Watchlist / Audit)**, and **Low Risk (Stable)**.

* **Decision & Downstream Action:**  
  * **Action:** Content operations allocates weekly editorial refresh bandwidth to top-scored URLs to update outdated information, refresh metadata, and re-optimize headers.  
  * **Counterfactual:** Without this model, content updates are either reactive (done weeks after impressions have already bottomed out) or arbitrary (updating healthy pages that don't need work).

* **Cost of a Wrong Call:**  
  * **False Positive (Predicting decay when stable):** Wasted editorial budget (~$200–$500 per in-depth refresh/rewrite) on content that would have sustained rank naturally.  
  * **False Negative (Missing a decaying URL):** Compounding loss of rank and conversions; re-ranking an abandoned page is significantly harder and more expensive than preserving an slipping one.

* **Why Data / ML is Needed (Why Not Simple Heuristics?):**  
  A simple rule like *"impressions dropped 10% this week"* triggers false alarms due to day-of-week seasonality, tracking glitches, or transient algorithmic tests by search engines. A machine learning model can aggregate multi-factor signals (query-level volatility, click CTR divergence from rank, and long-tail query loss) to separate temporary noise from structural, permanent decay.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [13]:
import os
from pathlib import Path

print("Current Working Directory:", os.getcwd())
print("\nFiles & Folders in current directory:")
print(os.listdir('.'))

# Check if data exists relative to where you are
for path in [Path("."), Path(".."), Path("../.."), Path("/content")]:
    matches = list(path.glob("**/*.parquet")) + list(path.glob("**/*.csv"))
    if matches:
        print(f"\nFound {len(matches)} data file(s) under {path.resolve()}:")
        for m in matches[:5]:
            print(" -", m)
        break
else:
    print("\nNo .parquet or .csv files found anywhere nearby.")

Current Working Directory: /content

Files & Folders in current directory:
['.config', 'sample_data']

Found 4 data file(s) under /content:
 - sample_data/california_housing_test.csv
 - sample_data/california_housing_train.csv
 - sample_data/mnist_train_small.csv
 - sample_data/mnist_test.csv


In [14]:
# Cell 1: Code - Dataset Exploration & Baseline Numbers
import os
from pathlib import Path
import numpy as np
import pandas as pd

# 1. Locate dataset dynamically (works both in local repo and Colab)
potential_roots = [Path("."), Path(".."), Path("../.."), Path("/content")]
data_files = []

for root in potential_roots:
    if root.exists():
        found = list(root.glob("**/*.parquet")) + list(root.glob("**/*.csv"))
        # Exclude Colab sample datasets and checkpoints
        found = [
            f
            for f in found
            if "sample_data" not in str(f) and ".ipynb_checkpoints" not in str(f)
        ]
        if found:
            data_files = found
            break

if not data_files:
    raise FileNotFoundError(
        "Could not find any .parquet or .csv files! "
        "If you are in Google Colab, upload your dataset using the folder icon on the left sidebar."
    )

data_file_path = data_files[0]
print(f"Loading data from: {data_file_path.resolve()}")

# 2. Read dataset
if data_file_path.suffix == ".parquet":
    df = pd.read_parquet(data_file_path)
else:
    df = pd.read_csv(data_file_path)

print(f"Loaded DataFrame with shape: {df.shape}")

# 3. Standardize column names (lowercase)
df.columns = [c.lower() for c in df.columns]

# 4. Compute baseline numbers
url_col = "url" if "url" in df.columns else ("page" if "page" in df.columns else None)
metric_col = (
    "impressions"
    if "impressions" in df.columns
    else ("clicks" if "clicks" in df.columns else None)
)

if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
    date_range_days = (df["date"].max() - df["date"].min()).days
else:
    date_range_days = "N/A"

total_records = len(df)
total_urls = df[url_col].nunique() if url_col else "N/A"

print("\n--- Key Baseline Metrics ---")
print(f"1. Total Records: {total_records:,}")
print(f"2. Unique URLs: {total_urls:,}" if url_col else "2. Unique URLs: N/A")
print(f"3. Date Range: {date_range_days} days")

# 5. Traffic concentration (80/20 rule validation)
if url_col and metric_col:
    top_10_cutoff = max(1, int(total_urls * 0.10))
    grouped = df.groupby(url_col)[metric_col].sum().sort_values(ascending=False)
    top_10_share = (grouped.iloc[:top_10_cutoff].sum() / grouped.sum()) * 100
    print(
        f"4. Traffic Concentration: Top 10% of URLs drive {top_10_share:.2f}% of total {metric_col}"
    )

Loading data from: /usr/local/lib/python3.13/dist-packages/pyogrio/tests/fixtures/list_field_values_file.parquet
Loaded DataFrame with shape: (5, 7)

--- Key Baseline Metrics ---
1. Total Records: 5
2. Unique URLs: N/A
3. Date Range: N/A days


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## 4. Careful Words: What I Can and Can't Claim

* **What I CAN claim:**  
  * I can identify statistical correlations and temporal lead indicators (e.g., declining CTR at stable average position) that precede sustained drops in visibility.
  * I can rank URLs by estimated risk to help content teams prioritize where human evaluation is most urgently needed.

* **What I CANNOT claim:**  
  * **Causality:** I cannot definitively claim *why* a page decayed (e.g., whether it was a Google Core Update, competitor backlink surge, or shifting user intent) without external SERP scraping.
  * **Guaranteed Lift:** I cannot promise that refreshing an identified page will reverse decay—only that the page is at risk and warrants editorial review.


## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.